In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("22-null-deep-dive")
dfs = register_views(spark)
emp  = dfs["employees"]
sal  = dfs["salary_history"]
pr   = dfs["performance_reviews"]
po   = dfs["purchase_orders"]
ee   = dfs["emp_events"]
lr   = dfs["leave_requests"]
dept = dfs["departments"]

# ── Section 1 ── NULL in INNER JOIN (rows dropped)
# Flaw: purchase_order 25 has dept_id=None
# INNER JOIN silently drops any row where the join key is NULL on either side
inner = emp.join(po, emp.dept_id == po.dept_id, "inner")
left  = emp.join(po, emp.dept_id == po.dept_id, "left")
print(f"emp={emp.count()}, po={po.count()}, INNER={inner.count()}, LEFT={left.count()}")
# PO order 25 has dept_id=None → INNER JOIN drops it; LEFT JOIN preserves emp rows


# ── Section 2 ── NULL in aggregations (skipped silently)
# Flaw: emp 10, 15 have salary=None; emp 19 has salary=0.0
# All aggregate functions (sum, avg, min, max) ignore NULLs by default
emp.agg(
    F.count("*").alias("total"),                                          # 41
    F.count("salary").alias("salary_non_null"),                           # 39
    F.sum("salary").alias("sum"),                                         # NULLs excluded from sum
    F.avg("salary").alias("avg_excl_null"),                               # avg of 39 non-null rows
    F.sum(F.coalesce(F.col("salary"), F.lit(0.0))).alias("sum_with_zero"),   # NULLs treated as 0
    F.avg(F.coalesce(F.col("salary"), F.lit(0.0))).alias("avg_with_zero"),   # avg of 41 rows
).show()
# Comment: avg_excl_null > avg_with_zero because null rows treated as 0 drag average down


# ── Section 3 ── NULL in window functions
# Flaw: salary_history boundary rows produce NULL from LAG/LEAD
# First row of each partition: lag=NULL (no prior row); last row: lead=NULL (no next row)
w = Window.partitionBy("emp_id").orderBy("effective_date")
sal.withColumn("lag_salary",  F.lag("salary_after",  1).over(w)) \
   .withColumn("lead_salary", F.lead("salary_after", 1).over(w)) \
   .select("emp_id", "effective_date", "salary_after", "lag_salary", "lead_salary") \
   .filter(F.col("emp_id") == 5).show()
# First row: lag=NULL (boundary); last row: lead=NULL (boundary)
# Dup rows for emp 5 on 2022-04-01: both have lag=132000, lead is the other dup's value


# ── Section 4 ── NULL in anti-join (NOT IN trap)
# Flaw: purchase_orders has dept_id=None (order 25)
# SQL NOT IN with a NULL in the subquery always returns 0 rows — a silent data loss bug
# SAFE: left_anti join — unaffected by NULLs on join key
emp.join(po.select("dept_id").distinct(), emp.dept_id == po.dept_id, "left_anti") \
   .select("emp_id", "dept_id").show(3)

# SQL NOT IN trap demonstration
spark.sql("""
    SELECT COUNT(*) AS cnt
    FROM employees
    WHERE dept_id NOT IN (SELECT dept_id FROM purchase_orders)
""").show()
# Expected: 0 because purchase_orders has a NULL dept_id (order 25)
# NULL in subquery causes entire NOT IN to evaluate to UNKNOWN for every row

# Correct approach: exclude NULLs from subquery
spark.sql("""
    SELECT COUNT(*) AS cnt
    FROM employees
    WHERE dept_id NOT IN (SELECT dept_id FROM purchase_orders WHERE dept_id IS NOT NULL)
""").show()


# ── Section 5 ── NULL in GROUP BY (creates its own group)
# Flaw: leave_request 11 has leave_type=None
# GROUP BY treats all NULLs as equal — they form their own group (SQL3 behaviour)
lr.groupBy("leave_type").count().show()
# leave_type=None is its own group (request 11, emp 34)
# This is correct SQL3 behaviour; NULLs are grouped together


# ── Section 6 ── NULL in ORDER BY (NULL sort position)
# Flaw: emp 10, 15 have salary=None
# Default: ASC → NULLs sort FIRST; DESC → NULLs sort LAST (Spark follows SQL standard)
emp.orderBy(F.col("salary").asc()).select("emp_id", "salary").show(5)             # NULLs first
emp.orderBy(F.col("salary").asc_nulls_last()).select("emp_id", "salary").show(5)  # NULLs last
emp.orderBy(F.col("salary").desc()).select("emp_id", "salary").show(5)            # NULLs last


# ── Section 7 ── NULL propagation in expressions
# Any arithmetic or string operation involving NULL produces NULL
from pyspark.sql.functions import lit
spark.createDataFrame([(None,), (5.0,), (0.0,)], ["x"]) \
     .withColumn("y", F.col("x") + 10) \
     .withColumn("z", F.col("x") * 2) \
     .withColumn("concat_r", F.concat(F.lit("val="), F.col("x").cast("string"))) \
     .show()
# NULL + 10 = NULL; NULL * 2 = NULL; concat with NULL = NULL (use concat_ws to skip NULLs)


# ── Section 8 ── NULL in SCD2 (coalesce for valid_to)
# Flaw: current-record rows in salary_history have valid_to=NULL after LEAD
# NULL valid_to = current record; coalesce maps NULL to far-future date for range filters
w_s = Window.partitionBy("emp_id").orderBy("effective_date")
sal.withColumn("valid_to", F.lead("effective_date", 1).over(w_s)) \
   .filter(
       (F.col("effective_date") <= F.lit("2021-01-01")) &
       (F.coalesce(F.col("valid_to"), F.lit("9999-12-31")) > F.lit("2021-01-01"))
   ).select("emp_id", "salary_after", "effective_date", "valid_to").show(10)
# NULLs in valid_to = current records; coalesce maps them to a far-future date


# ── Section 9 ── Null-safe equality (eqNullSafe)
# Flaw: salary_history rows 15,16 are exact dups; salary_before=None on initial hire rows
# Normal == : NULL == NULL evaluates to NULL (not True) → those rows fall through filters
# eqNullSafe: NULL <=> NULL evaluates to True
sal.filter(F.col("salary_before").eqNullSafe(F.col("salary_after"))).show()
# eqNullSafe: NULL eqNullSafe NULL = True; normal == returns NULL for NULL comparisons
# Surfaces initial-hire rows where salary_before IS NULL and also catches zero-change rows


# ── Section 10 ── Complete NULL audit (summary per table)
# Surfaces every NULL across the entire dataset — important for data quality baseline
for name, df in dfs.items():
    null_counts = {c: df.filter(F.col(c).isNull()).count()
                   for c in df.columns if df.schema[c].nullable}
    has_nulls = {c: n for c, n in null_counts.items() if n > 0}
    if has_nulls:
        print(f"\n{name}: NULLs found → {has_nulls}")


stop_and_wait(spark)